# M4 IA Agética - Adicionando uma avaliação em nível de componente ao fluxo de trabalho de pesquisa

## 1. Introdução

No laboratório avaliado anterior (M3), você construiu um agente de pesquisa que utilizava ferramentas e executava um fluxo de trabalho de três etapas:

1. Buscar informações na web.

2. Refletir sobre o resultado.

3. Publicar um relatório HTML claro.

Agora, neste laboratório não avaliado, você se concentrará em avaliar **um componente desse fluxo de trabalho**: a *etapa de pesquisa*.

Em vez de gerar ensaios e refiná-los, aqui você projetará uma **avaliação em nível de componente** para verificar a qualidade das fontes retornadas pela etapa de pesquisa.

A avaliação comparará os URLs recuperados pelo agente com uma **lista predefinida de domínios preferenciais** (por exemplo, `arxiv.org`, `nature.com`, `nasa.gov`).

Isso permite quantificar se o sistema está obtendo informações de fontes confiáveis, usando uma **avaliação objetiva, exemplo a exemplo, da verdade fundamental**.

### 1.1. Visão geral do laboratório

No vídeo, Andrew mostrou um caso em que os resultados da pesquisa na web eram de **baixa qualidade**, dificultando a confiança nas informações recuperadas.

Com base nesse exemplo, neste laboratório você avaliará a confiabilidade das fontes comparando-as com uma **lista predefinida de domínios preferenciais**.

Para esta avaliação, focaremos no tópico *“desenvolvimentos recentes na ciência dos buracos negros”*, um dos exemplos destacados no curso.

A ideia é verificar se a ferramenta de busca na web está retornando fontes de domínios preferenciais e quantificar a proporção de resultados preferenciais em relação ao total de resultados.

Esta avaliação será implementada como uma única função que realiza uma **verificação objetiva, exemplo a exemplo**. Ela irá:

* Analisar a saída do Tavily (nossa ferramenta de busca na web). * Identificar quais URLs pertencem à lista de **domínios preferenciais**.

* Calcular a proporção de fontes preferenciais em relação ao total de fontes recuperadas.

* Retornar um indicador booleano (**APROVADO/REPROVADO**) e um resumo formatado em Markdown que pode ser incorporado diretamente em relatórios.

<img src="M4-UGL-1.png" width="70%">

### 1.2. 🎯 Resultados de aprendizagem

Você aprenderá a:

* Escrever uma função que verifica os resultados de pesquisa de uma API de pesquisa na web em busca de **fontes preferenciais**.

* Criar uma avaliação para verificar se suas fontes vêm de seus **domínios preferenciais**.

* Adicionar uma **avaliação em nível de componente** à função de pesquisa na web.

## 2. Configuração: Importar bibliotecas e carregar o ambiente

Como nos laboratórios anteriores, você começará importando as bibliotecas necessárias e inicializando seu ambiente.

In [ ]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import json
import re

# --- Third-party ---
from aisuite import Client

# --- Local / project ---
import research_tools
import utils

client = Client()

## 3. Etapa de Pesquisa – `find_references`

No laboratório avaliado, a função que você implementou **pesquisava na web e escrevia um rascunho de relatório** em uma única etapa.

Aqui, separamos a funcionalidade de pesquisa na web em uma função independente chamada `find_references`. Isso permite que você avalie os resultados da pesquisa independentemente das etapas de escrita e reflexão, que omitiremos deste laboratório, já que estamos focados apenas no resultado da etapa de pesquisa na web.

Observe duas diferenças principais em relação à implementação do laboratório avaliado:

* Esta nova função utiliza o **AISuite**, que gerencia automaticamente as chamadas de ferramentas para você (em vez de escrever código manual para chamar ferramentas com o SDK da OpenAI).

* A função também informa ao LLM a **data atual**, o que ajuda a melhorar a relevância para consultas sensíveis ao tempo.

O papel de `find_references` é **coletar informações externas** de ferramentas como **Arxiv**, **Tavily** e **Wikipedia**. Como a qualidade desses resultados influencia diretamente os resultados do laboratório avaliado, esta é a etapa em que você pode aplicar **métodos de avaliação** — por exemplo, verificando se os URLs retornados pertencem à sua lista de **domínios preferenciais**.

In [ ]:
def find_references(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """Perform a research task using external tools (arxiv, tavily, wikipedia)."""

    prompt = f"""
    You are a research function with access to:
    - arxiv_tool: academic papers
    - tavily_tool: general web search (return JSON when asked)
    - wikipedia_tool: encyclopedic summaries

    Task:
    {task}

    Today is {datetime.now().strftime('%Y-%m-%d')}.
    """.strip()

    messages = [{"role": "user", "content": prompt}]
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool,
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            max_turns=5,
        )
        content = response.choices[0].message.content
        return (content, messages) if return_messages else content
    except Exception as e:
        return f"[Model Error: {e}]"

Execute a seguinte célula para testar a função de pesquisa.
Esta tarefa irá recuperar dois artigos recentes sobre desenvolvimentos na ciência dos buracos negros e exibir os resultados.

In [ ]:
research_task = "Find 2 recent papers about recent developments in black hole science"
research_result = find_references(research_task)

utils.print_html(
    research_result,
    title="Research Function Output"
)

## 4. Etapa de Avaliação – Domínios Preferenciais

Nem todas as fontes recuperadas por buscas na web são igualmente confiáveis.

Neste laboratório, focaremos em **apenas uma etapa do laboratório avaliado anterior** — a etapa de pesquisa `find_references` — e mostraremos como projetar uma **avaliação em nível de componente** que verifica se os domínios retornados pertencem a uma lista predefinida de **domínios preferenciais**.

Este é um exemplo de uma **avaliação objetiva com uma verdade fundamental clara para cada exemplo**.

Relembrando a aula, recordamos os dois eixos de avaliação: ao longo desses eixos, estamos trabalhando no **quadrante superior esquerdo** — avaliações objetivas com verdade fundamental explicitamente definida, aplicada no nível de cada exemplo.

<img src='M4-UGL-1-Evaluations.png' width='80%'>

### Por que avaliações em nível de componente?

Como Andrew mencionou na aula:

- Se o problema estiver na busca na web (geralmente o **primeiro passo** em um fluxo de trabalho de laboratório avaliado), executar *todo* o pipeline (busca → rascunho → reflexão) todas as vezes pode ser **caro** e gerar ruído.

- Pequenas melhorias na qualidade da busca na web podem ser mascaradas pela aleatoriedade introduzida por componentes posteriores.

- Ao avaliar a busca na web *sozinha*, você obtém um **sinal mais claro** de se esse componente está melhorando.

Avaliações em nível de componente também são eficientes quando várias equipes trabalham em diferentes partes de um sistema: cada equipe pode otimizar seu próprio componente usando uma métrica clara, sem precisar executar ou esperar por testes completos de ponta a ponta.

### Como avaliamos?

Nossa avaliação aqui é **objetiva** e, portanto, pode ser avaliada usando código. Ela possui uma verdade fundamental específica do exemplo - a lista de fontes preferenciais para esta consulta de "buraco negro". Para construir a avaliação, você deverá:

1. Extrair os URLs retornados pelo Tavily. 2. Compare-os com uma lista predefinida de **domínios preferenciais** (por exemplo, `arxiv.org`, `nature.com`, `nasa.gov`).

3. Calcule a **proporção de resultados preferenciais em relação ao total de resultados**.

4. Retorne um indicador **APROVADO/REPROVADO** juntamente com um resumo formatado em Markdown.

Isso fornece uma métrica reproduzível e de baixo custo que nos informa se o componente de pesquisa — e somente esta etapa do laboratório avaliado — está utilizando fontes confiáveis.

In [ ]:
# list of preferred domains for Tavily results
TOP_DOMAINS = {
    # General reference / institutions / publishers
    "wikipedia.org", "nature.com", "science.org", "sciencemag.org", "cell.com",
    "mit.edu", "stanford.edu", "harvard.edu", "nasa.gov", "noaa.gov", "europa.eu",

    # CS/AI venues & indexes
    "arxiv.org", "acm.org", "ieee.org", "neurips.cc", "icml.cc", "openreview.net",

    # Other reputable outlets
    "elifesciences.org", "pnas.org", "jmlr.org", "springer.com", "sciencedirect.com",

    # Extra domains (case-specific additions)
    "pbs.org", "nova.edu", "nvcc.edu", "cccco.edu",

    # Well known programming sites
    "codecademy.com", "datacamp.com"
}

def evaluate_tavily_results(TOP_DOMAINS, raw: str, min_ratio=0.4):
    """
    Evaluate whether plain-text research results mostly come from preferred domains.

    Args:
        TOP_DOMAINS (set[str]): Set of preferred domains (e.g., 'arxiv.org', 'nature.com').
        raw (str): Plain text or Markdown containing URLs.
        min_ratio (float): Minimum preferred ratio required to pass (e.g., 0.4 = 40%).

    Returns:
        tuple[bool, str]: (flag, markdown_report)
            flag -> True if PASS, False if FAIL
            markdown_report -> Markdown-formatted summary of the evaluation
    """

    # Extract URLs from the text
    url_pattern = re.compile(r'https?://[^\s\]\)>\}]+', flags=re.IGNORECASE)
    urls = url_pattern.findall(raw)

    if not urls:
        return False, """### Evaluation — Tavily Preferred Domains
No URLs detected in the provided text. 
Please include links in your research results.
"""

    # Count preferred vs total
    total = len(urls)
    preferred_count = 0
    details = []

    for url in urls:
        domain = url.split("/")[2]
        preferred = any(td in domain for td in TOP_DOMAINS)
        if preferred:
            preferred_count += 1
        details.append(f"- {url} → {'✅ PREFERRED' if preferred else '❌ NOT PREFERRED'}")

    ratio = preferred_count / total if total > 0 else 0.0
    flag = ratio >= min_ratio

    # Markdown report
    report = f"""
### Evaluation — Tavily Preferred Domains
- Total results: {total}
- Preferred results: {preferred_count}
- Ratio: {ratio:.2%}
- Threshold: {min_ratio:.0%}
- Status: {"✅ PASS" if flag else "❌ FAIL"}

**Details:**
{chr(10).join(details)}
"""
    return flag, report

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#dbeafe; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
<strong>🔎 Por que esta é uma avaliação objetiva:</strong><br><br>
Cada URL recuperada do Tavily é comparada a uma lista predefinida de <em>domínios preferenciais</em> (<code>TOP_DOMAINS</code>):<br>
• Se o domínio corresponder → ✅ PREFERENCIAL<br>
• Caso contrário → ❌ NÃO PREFERENCIAL<br><br>
Isso gera um sinal claro de APROVADO/REPROVADO, dependendo da proporção de fontes preferenciais excede um determinado limite.
Como a verdade fundamental (preferido vs. não preferido) é definida explicitamente para cada exemplo, a avaliação é objetiva e reproduzível.

</div>

Execute a célula para exibir os domínios preferenciais de amostra, os resultados da pesquisa e o resumo da avaliação (APROVADO/REPROVADO com detalhes).

In [ ]:
utils.print_html(json.dumps(list(TOP_DOMAINS)[:4], indent=2), title="Sample Trusted Domains")

utils.print_html("<h3>Research Results</h3>" + research_result, title="Research Results")

flag, report = evaluate_tavily_results(TOP_DOMAINS, research_result)
utils.print_html("<pre>" + report + "</pre>", title="<h3>Evaluation Summary</h3>")

## Experimente você mesmo!

Agora é a sua vez.

Nesta seção, você pode experimentar diretamente a **etapa de pesquisa** e sua **avaliação**:

* **Tópico**: escolha um tópico diferente para pesquisar.

* **Domínios preferenciais**: edite ou expanda a lista `TOP_DOMAINS`.

* **Taxa de avaliação**: ajuste a `min_ratio` (por exemplo, 0,4 = pelo menos 40% de fontes preferenciais).

Execute novamente as células abaixo após fazer suas edições para ver como a avaliação muda.

In [ ]:
# === 5.1. Try it yourself: topic, ratio & preferred domains ===
# Edit these parameters before running the cell

topic = "recent developments in black hole science"   # <- Change the topic here
min_ratio = 0.4                                       # <- Change threshold (0.0–1.0)
run_reflection = True                                 # <- Set False to skip Step 4

# Short list of preferred domains (edit or expand as needed)
TOP_DOMAINS = {
    "wikipedia.org", "nature.com", "science.org", "arxiv.org",
    "nasa.gov", "mit.edu", "stanford.edu", "harvard.edu"
}

# Show a sample of preferred domains
import json
utils.print_html(
    json.dumps(sorted(list(TOP_DOMAINS)), indent=2),
    title="<h3>Sample Preferred Domains</h3>"
)

# 1) Research
research_task = f"Find 2–3 key papers and reliable overviews about {topic}."
research_output = find_references(research_task)
utils.print_html(research_output, title=f"<h3>Research Results on {topic}</h3>")

# 2) Evaluate sources (preferred domains ratio)
flag, eval_md = evaluate_tavily_results(TOP_DOMAINS, research_output, min_ratio=min_ratio)
utils.print_html("<pre>" + eval_md + "</pre>", title="<h3>Evaluation Summary</h3>")

## 5. Principais Conclusões

* Você acabou de ver como avaliar o desempenho de **um componente**: a etapa de pesquisa `find_references`.

* Sua avaliação em nível de componente verificou se os URLs recuperados estavam em uma lista predefinida de **domínios preferenciais**.

* Este é um exemplo de **avaliação objetiva** com uma **verdade fundamental clara por exemplo**.

* Para criar um conjunto de avaliação, você pode elaborar cerca de 10 perguntas abrangendo diferentes tópicos (astronomia, robótica, finanças, etc.) e definir domínios preferenciais para cada uma.

* A porcentagem de fontes recuperadas que correspondem à lista de domínios preferenciais fornece uma **métrica** útil para orientar melhorias, como o ajuste das perguntas ou dos parâmetros da ferramenta.

* Essa abordagem é **mais simples e barata** do que avaliar redações completas com reflexão e reescritas, já que você se concentra apenas no componente de busca na web.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 **Parabéns!**

Você projetou uma **avaliação em nível de componente** que torna seu agente de pesquisa mais confiável.

Ao verificar diretamente a qualidade das fontes, você introduziu uma salvaguarda que é **objetiva, reproduzível e econômica**.

Isso está alinhado com a ideia destacada na palestra de Andrew: *avaliações em nível de componente* permitem testar partes individuais de um sistema de IA sem a sobrecarga de avaliar todo o pipeline.

</div>